# Notebook 1: Municipal codes covering census years 1980-2020

Created: 2026-04-21

### The Task

I want to derive commuter data from the Population Census to produce CSV files in the same format and structure as in <https://github.com/daisukeadachi/commuting_zone_japan/tree/main/data>. Here are a few lines:

```
living_mun,commute_mun,pop,tot_pop_living_mun,year
1101,1101,53411,76584,2015
1101,1102,4225,76584,2015
1101,1103,2795,76584,2015
```

The 5 fields in this CSV table are the following:

<dl>
<dt>living_mun</dt>
<dd>The municipal code for the commuters' residences.</dd>
<dt>commute_mun</dt>
<dd>The municipal code for the commuters' places of work or school</dd>
<dt>pop</dt>
<dd>The number of people who make this commute.</dd>
<dt>tot_pop_living_mun</dt>
<dd>The total commuter population of the residential municipality.</dd>
<dt>year</dt>
<dd>Census year</dd>
</dl>

In order to iterate through the databases by municipality, I need the municipal codes for census years 1980-2020. Municipal code tables are available in Japanese and English for years from 1980 through 2023. Here I select census years and create a dataset of codes, Japanese names, and English names. See <https://estatjp.readthedocs.io/en/latest/tutorials/getting-started.html> for a walk-through of getting API urls for databases in e-Stat. Here are the specific links for the databases accessed in this task:

<dl>
<dt>Municipal codes in English</dt>
<dd><a href="https://www.e-stat.go.jp/en/dbview?sid=0000020101">https://www.e-stat.go.jp/en/dbview?sid=0000020101</a> (Note: When this link is first clicked, e-Stat may respond with a "page not found" message. Reloading the page brings up the database page.)</dd>
<dt>Municipal codes in Japanese</dt>
<dd><a href="https://www.e-stat.go.jp/dbview?sid=0000020101">https://www.e-stat.go.jp/dbview?sid=0000020101</a></dd>
</dl>

In [ ]:
# For the municipal codes in English from https://www.e-stat.go.jp/en/dbview?sid=0000020101
codeapiurlen = "http://api.e-stat.go.jp/rest/3.0/app/getSimpleStatsData?cdCat01=A1101&cdTime=1980100000%2C1985100000%2C1990100000%2C1995100000%2C2000100000%2C2005100000%2C2010100000%2C2015100000%2C2020100000&appId=&lang=E&statsDataId=0000020101&metaGetFlg=Y&cntGetFlg=N&explanationGetFlg=Y&annotationGetFlg=Y&sectionHeaderFlg=1&replaceSpChars=0"

# from https://www.e-stat.go.jp/dbview?sid=0000020101
codeapiurlja = "http://api.e-stat.go.jp/rest/3.0/app/getSimpleStatsData?cdCat01=A1101&cdTime=1980100000%2C1985100000%2C1990100000%2C1995100000%2C2000100000%2C2005100000%2C2010100000%2C2015100000%2C2020100000&appId=&lang=J&statsDataId=0000020101&metaGetFlg=Y&cntGetFlg=N&explanationGetFlg=Y&annotationGetFlg=Y&sectionHeaderFlg=1&replaceSpChars=0"

# from https://www.e-stat.go.jp/dbview?sid=0003410374
apiurl2015 = "http://api.e-stat.go.jp/rest/3.0/app/getSimpleStatsData?cdArea=01101&cdCat02=110%2C120&appId=&lang=J&statsDataId=0003410374&metaGetFlg=Y&cntGetFlg=N&explanationGetFlg=Y&annotationGetFlg=Y&sectionHeaderFlg=1&replaceSpChars=0"


### Set up for coding

Do the following in a terminal and in the Python .venv environment.

```
pip install python-dotenv
dotenv set ESTAT_APP_ID your-app-id
pip install estatjp
```

### Fetch the codes with municipality names in English

In [3]:
import datetime
from estatjp import api
from estatjp import exceptions as xs
import pandas as pd
try:
    dfs_en = api.get_csv_data(codeapiurlen,description=datetime.datetime.now())
except Exception as e:
    print(type(e))
    print(e.args)
    print(e.with_traceback)

print(dfs_en.get('Main'))

       tab_code  Observation Value cat01_code  \
0             1  Observation value      A1101   
1             1  Observation value      A1101   
2             1  Observation value      A1101   
3             1  Observation value      A1101   
4             1  Observation value      A1101   
...         ...                ...        ...   
25058         1  Observation value      A1101   
25059         1  Observation value      A1101   
25060         1  Observation value      A1101   
25061         1  Observation value      A1101   
25062         1  Observation value      A1101   

               A Population and Households  area_code  \
0      A1101_Total population (Both sexes)       1100   
1      A1101_Total population (Both sexes)       1100   
2      A1101_Total population (Both sexes)       1100   
3      A1101_Total population (Both sexes)       1100   
4      A1101_Total population (Both sexes)       1100   
...                                    ...        ...   
25058  A1101

### Select the columns I want.

The municipal codes are in column 'area_code' that is transmitted as an integer when it is actually a 5-letter string. The first 2 digits make up the prefecture code. The next 3 digits make up the municipal code. (An English description of the census district coding system as of the 2020 census is at <https://www.stat.go.jp/english/data/kokusei/2020/pdf/exp.pdf>.) This code follows a tutorial on <a href="https://www.geeksforgeeks.org/python/python-pandas-int-to-string-with-leading-zeros/#convert-integers-to-strings-in-pandas-dataframe-with-leading-zeros">GeeksForGeeks</a> to convert it back to a zero-filled string.

In [4]:
maindf_en = dfs_en.get('Main')
maindf_en['area_code'] = maindf_en['area_code'].apply(lambda x: f"{x:05}")
maindf_en[["SURVEY YEAR","time_code","area_code","AREA","value"]]

,SURVEY YEAR,time_code,area_code,AREA,value
0,1980,1980100000,01100,Hokkaido Sapporo-shi,1401757
1,1985,1985100000,01100,Hokkaido Sapporo-shi,1542979
2,1990,1990100000,01100,Hokkaido Sapporo-shi,1671742
3,1995,1995100000,01100,Hokkaido Sapporo-shi,1757025
4,2000,2000100000,01100,Hokkaido Sapporo-shi,1822368
...,...,...,...,...,...
25058,2000,2000100000,47382,Okinawa-ken Yonaguni-cho,1852
25059,2005,2005100000,47382,Okinawa-ken Yonaguni-cho,1796
25060,2010,2010100000,47382,Okinawa-ken Yonaguni-cho,1657
25061,2015,2015100000,47382,Okinawa-ken Yonaguni-cho,1843


### Fetch the codes with municipality names in Japanese

In [5]:
try:
    dfs_ja = api.get_csv_data(codeapiurlja,description=datetime.datetime.now())
except Exception as e:
    print(type(e))
    print(e.args)
    print(e.with_traceback)

print(dfs_ja.get('Main'))

       tab_code  観測値 cat01_code    Ａ　人口・世帯  area_code        地域   time_code  \
0             1  観測値      A1101  A1101_総人口       1100   北海道 札幌市  1980100000   
1             1  観測値      A1101  A1101_総人口       1100   北海道 札幌市  1985100000   
2             1  観測値      A1101  A1101_総人口       1100   北海道 札幌市  1990100000   
3             1  観測値      A1101  A1101_総人口       1100   北海道 札幌市  1995100000   
4             1  観測値      A1101  A1101_総人口       1100   北海道 札幌市  2000100000   
...         ...  ...        ...        ...        ...       ...         ...   
25058         1  観測値      A1101  A1101_総人口      47382  沖縄県 与那国町  2000100000   
25059         1  観測値      A1101  A1101_総人口      47382  沖縄県 与那国町  2005100000   
25060         1  観測値      A1101  A1101_総人口      47382  沖縄県 与那国町  2010100000   
25061         1  観測値      A1101  A1101_総人口      47382  沖縄県 与那国町  2015100000   
25062         1  観測値      A1101  A1101_総人口      47382  沖縄県 与那国町  2020100000   

          調査年 unit    value  annotation  
0      19

In [6]:
maindf_ja = dfs_ja.get('Main')
maindf_ja['area_code'] = maindf_en['area_code'].apply(lambda x: f"{x:05}")
maindf_ja[["調査年","time_code","area_code","地域","value"]]

,調査年,time_code,area_code,地域,value
0,1980年度,1980100000,01100,北海道 札幌市,1401757
1,1985年度,1985100000,01100,北海道 札幌市,1542979
2,1990年度,1990100000,01100,北海道 札幌市,1671742
3,1995年度,1995100000,01100,北海道 札幌市,1757025
4,2000年度,2000100000,01100,北海道 札幌市,1822368
...,...,...,...,...,...
25058,2000年度,2000100000,47382,沖縄県 与那国町,1852
25059,2005年度,2005100000,47382,沖縄県 与那国町,1796
25060,2010年度,2010100000,47382,沖縄県 与那国町,1657
25061,2015年度,2015100000,47382,沖縄県 与那国町,1843


### Merge these two tables

In [7]:
import pandas as pd
result = pd.merge(maindf_en[["SURVEY YEAR","time_code","area_code","AREA","value"]],maindf_ja[["調査年","time_code","area_code","地域","value"]], on=["time_code","area_code","value"] )
result

,SURVEY YEAR,time_code,area_code,AREA,value,調査年,地域
0,1980,1980100000,01100,Hokkaido Sapporo-shi,1401757,1980年度,北海道 札幌市
1,1985,1985100000,01100,Hokkaido Sapporo-shi,1542979,1985年度,北海道 札幌市
2,1990,1990100000,01100,Hokkaido Sapporo-shi,1671742,1990年度,北海道 札幌市
3,1995,1995100000,01100,Hokkaido Sapporo-shi,1757025,1995年度,北海道 札幌市
4,2000,2000100000,01100,Hokkaido Sapporo-shi,1822368,2000年度,北海道 札幌市
...,...,...,...,...,...,...,...
25058,2000,2000100000,47382,Okinawa-ken Yonaguni-cho,1852,2000年度,沖縄県 与那国町
25059,2005,2005100000,47382,Okinawa-ken Yonaguni-cho,1796,2005年度,沖縄県 与那国町
25060,2010,2010100000,47382,Okinawa-ken Yonaguni-cho,1657,2010年度,沖縄県 与那国町
25061,2015,2015100000,47382,Okinawa-ken Yonaguni-cho,1843,2015年度,沖縄県 与那国町


### Select the desired columns and rename them

In [8]:
df_output = result[["SURVEY YEAR","time_code","area_code","AREA","地域","value"]]
df_output.columns = ['survey_year','time_code','muni_code','muni_name_en','muni_name_ja','population']
df_output

,survey_year,time_code,muni_code,muni_name_en,muni_name_ja,population
0,1980,1980100000,01100,Hokkaido Sapporo-shi,北海道 札幌市,1401757
1,1985,1985100000,01100,Hokkaido Sapporo-shi,北海道 札幌市,1542979
2,1990,1990100000,01100,Hokkaido Sapporo-shi,北海道 札幌市,1671742
3,1995,1995100000,01100,Hokkaido Sapporo-shi,北海道 札幌市,1757025
4,2000,2000100000,01100,Hokkaido Sapporo-shi,北海道 札幌市,1822368
...,...,...,...,...,...,...
25058,2000,2000100000,47382,Okinawa-ken Yonaguni-cho,沖縄県 与那国町,1852
25059,2005,2005100000,47382,Okinawa-ken Yonaguni-cho,沖縄県 与那国町,1796
25060,2010,2010100000,47382,Okinawa-ken Yonaguni-cho,沖縄県 与那国町,1657
25061,2015,2015100000,47382,Okinawa-ken Yonaguni-cho,沖縄県 与那国町,1843


### Write to file

In [9]:
df_output.to_csv("../data/municipal-codes.csv", index=False)